<a href="https://colab.research.google.com/github/goutham3010/Hospital-Patient-Flow-and-Resource-Demand-Prediction-System/blob/main/02_Exploratory_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOSPITAL PATIENT FLOW & DEMAND PREDICTION SYSTEM
## NOTEBOOK 02: EXPLORATORY DATA ANALYSIS (EDA)
**Workflow Phase 02** of the 17-Stage Hospital Operations Decision Support Architecture.

### Objectives:
1. Analyze longitudinal admission and discharge trends over time (daily, monthly, yearly).
2. Examine day-of-week seasonality (weekday surgery schedules vs. weekend emergency intake).
3. Evaluate Length of Stay (LOS) distributions across hospital departments.
4. Calculate hospital bed census and active capacity utilization dynamics.
5. Export structured EDA summaries for dashboard consumption (`data/eda_summary.json`).


In [ ]:
# 1. IMPORT LIBRARIES & CONFIGURE PLOT STYLES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print("Libraries imported successfully.")


In [ ]:
# 2. LOAD DATASET & PARSE TEMPORAL ATTRIBUTES
df = pd.read_csv('admissions.csv')
df['admit_date'] = pd.to_datetime(df['admit_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

df['admit_year'] = df['admit_date'].dt.year
df['admit_month'] = df['admit_date'].dt.to_period('M')
df['admit_dow'] = df['admit_date'].dt.day_name()
print("Temporal attributes parsed.")


In [ ]:
# 3. DAY-OF-WEEK ADMISSION SURGE PATTERNS
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
plt.figure(figsize=(10, 4.5))
ax = sns.countplot(data=df, x='admit_dow', order=dow_order, palette='crest')
plt.title("Total Hospital Admissions by Day of the Week (2015 - 2024)", fontsize=13, fontweight='bold')
plt.xlabel("Day of the Week", fontsize=11)
plt.ylabel("Total Admissions", fontsize=11)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height() - 1200),
                ha='center', va='center', color='white', fontweight='bold', fontsize=10)
plt.show()


In [ ]:
# 4. DEPARTMENTAL WARD DISTRIBUTION
plt.figure(figsize=(7, 7))
ward_counts = df['ward_type'].value_counts()
colors = ['#06b6d4', '#ef4444', '#f59e0b', '#a855f7']
plt.pie(ward_counts, labels=ward_counts.index, autopct='%1.1f%%', startangle=140, colors=colors,
        explode=(0.04, 0.04, 0.04, 0.04), textprops={'fontsize': 11, 'fontweight': 'bold'})
plt.title("Distribution of Inpatient Admissions across Hospital Wards", fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# 5. LENGTH OF STAY (LOS) DISTRIBUTION BY WARD
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='ward_type', y='los_days', palette='Set2', showmeans=True)
plt.title("Length of Stay (Days) across Hospital Departments", fontsize=13, fontweight='bold')
plt.xlabel("Hospital Ward", fontsize=11)
plt.ylabel("Length of Stay (Days)", fontsize=11)
plt.ylim(0, 25)
plt.show()

print("Mean LOS by Ward:")
print(df.groupby('ward_type')['los_days'].mean().round(2))


In [ ]:
# 6. LONGITUDINAL MONTHLY ADMISSION TRAJECTORY
monthly_counts = df.groupby('admit_month').size()
plt.figure(figsize=(14, 5))
monthly_counts.plot(color='#0284c7', linewidth=1.8)
plt.title("Longitudinal Monthly Admission Volume (120 Months: 2015 - 2024)", fontsize=13, fontweight='bold')
plt.xlabel("Year - Month", fontsize=11)
plt.ylabel("Monthly Admissions", fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# 7. GENERATE AND SAVE EDA ARTIFACTS
import sys
sys.path.append('src')
from eda import compute_eda_summary, save_eda_artifacts

summary = compute_eda_summary(df)
save_eda_artifacts(summary, 'data/eda_summary.json')
print("EDA artifacts saved to data/eda_summary.json")
